# Fully Connected Neural Network: Backward Pass

In this notebook, you will:
- Implement the backward pass for a fully connected layer, the sigmoid activation function, and a fully connected network;
- Write a loss function and its gradient;
- Implement the code for training the network.

By the end of this assignment, you will have a complete implementation of a fully connected neural network that can be trained for various tasks!

In [ ]:
# Ensuring that the tests in the notebook work correctly
! pip install numpy==1.22.4

In [ ]:
import numpy as np
np.random.seed(42)

## Implementation of parameter updates for layers and the neural network

We will start by implementing parameter update methods for the fully connected layer, the sigmoid activation function, and the neural network.

### Task 1: Backward Pass for a Fully Connected Layer

In order to implement backpropagation, we need to be able to compute gradients of the loss function by weights of the layer. Let's remind ourselves of how to compute them.

The formula of the linear layer: $$Z = XW$$

We know gradient of the loss function by the output $\frac{\partial L}{\partial Z}$.

And we need to get:
$$\frac{\partial L}{\partial X} = ?$$

First of all, let's recall what are these objects:

* $\frac{\partial L}{\partial Z}$ is the gradient of $L$ (scalar) by the activation $Z$ (row vector of size (1, output_size)). Hence, $\frac{\partial L}{\partial Z}$ is also row vector of size (1, output_size).
* $\frac{\partial L}{\partial X}$ is a row vector of size (1, input_size).
* $W$ is a matrix of shape (input_size, output_size).

Now, let's calculate

$$\frac{\partial L}{\partial X_{ij}} = \sum_{k,s}\frac{\partial L}{\partial Z_{ks}}\cdot \frac{\partial Z_{ks}}{\partial X_{ij}}$$

Ok, we already have the first multiple. What about the second? To get it, let's express $Z_{ks}$ as a function of $X_{ij}$:

$$Z_{ks} = \sum_jX_{kj}W_{js}$$

You see that not all $X_{ij}$ are encountered here; only those with $i = k$. Namely:

$$\frac{\partial Z_{ks}}{\partial X_{ij}} = \begin{cases}
W_{js},\mbox{ if $i = k$},\\
0,\mbox{ otherwise}.
\end{cases}$$

So, let's return now to the sum
$$\frac{\partial L}{\partial X_{ij}} = \sum_{k,s}\frac{\partial L}{\partial Z_{ks}}\cdot \frac{\partial Z_{ks}}{\partial X_{ij}}$$

From what we discuss here, it follows that only summands with $k = i$ are left:
$$\frac{\partial L}{\partial X_{ij}} = \sum_{s}\frac{\partial L}{\partial Z_{is}}\cdot \frac{\partial Z_{is}}{\partial X_{ij}}=
\sum_{s}\frac{\partial L}{\partial Z_{is}}\cdot W_{js}$$

Let's now rewrite it like this:

$$\ldots = \sum_{s}\frac{\partial L}{\partial Z_{is}}\cdot (W^T)_{sj}$$

Do you recognize it? That's exactly

$$\left(\frac{\partial L}{\partial Z}\cdot W^T\right)_{ij}$$

So, finally
$$\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Z}\cdot W^T$$

And a little note about dimensions:

$$\underbrace{\frac{\partial L}{\partial X}}_{\mbox{(1, input_size)}} = \underbrace{\frac{\partial L}{\partial Z}}_{\mbox{(1, output_size)}}\cdot \underbrace{W^T}_{\mbox{(output_size, input_size)}},$$
which is valid.

Also compare this with the initial product:

$$\underbrace{Z}_{\mbox{(1, output_size)}} = \underbrace{X}_{\mbox{(1, input_size)}}\cdot \underbrace{W}_{\mbox{(input_size, output_size)}}$$

**Note**. In just the same way you can derive that

$$\frac{\partial L}{\partial W} = X^T\cdot\frac{\partial L}{\partial Z}$$

First, let's define the `Module` class, which will be the parent class for all layer and network classes:

In [ ]:
class Module():
    '''
    Abstract parent class for all layer and activation function classes.
    '''

    def __init__(self):
        '''
        Method called when an instance of the class is created.
        Parameters of layers will be defined here.
        '''
        pass

    def forward(self, x):
        '''
        Method implementing the forward pass of the network. That is,
        this method defines how input data is transformed into the output.

        params:
            x (np.ndarray) — input to the layer
        returns:
            out (np.ndarray) — output of the layer after all transformations
        '''
        return x

    def __call__(self, x):
        '''
        This method is called when an instance of the `Module` class is called.
        For example:
            model = Module()
            model(x)
        When `model(x)` is called, the `__call__` method is invoked.
        Here, we specify that calling `__call__` will call the `forward` method,
        meaning `model(x)` is equivalent to `model.forward(x)`.
        '''
        return self.forward(x)

    def backward(self, x, grad_output):
        '''
        Method implementing the backward pass of the network.
        In this method, the layer is trained. We will implement it later in the course
        after discussing how neural networks are trained.
        '''
        return grad_output

Now, let's define the `Linear` class. You need to fill in the missing parts in the `__init__` and `forward` methods with the code you implemented in earlier tasks, as well as implement the `backward` method.

**Note:** In the `forward` method, we added a line to save the input vector into the variable `self.x`. This will be useful for computing the gradients of the layer's weights in the `backward` method.

**Note:** In the class below the `backward` method does both gradient computations and a gradient step, unlike how it's implemented in Pytorch.

**A math exercise for you:** Derive how to get $\frac{\partial L}{\partial b}$ from $\frac{\partial L}{\partial X}$. If you're struggling, you can check in the slides.

In [ ]:
class Linear(Module):

    '''
    Class implementing a fully connected network layer
    '''

    def __init__(self, in_features, out_features, bias=True):
        '''
        Method to declare the weight matrices of the layer

        params:
            in_features (int): number of input neurons in the layer
                               (first dimension of the weight matrix W)
            out_features (int): number of output neurons in the layer
                                (second dimension of the weight matrix W)
            bias (bool): whether the layer has a bias vector:
                         - if bias=True, the layer output is computed as y=XW+b
                         - if bias=False, the layer output is computed as y=XW
        '''

        # Define the layer's weight matrix W. Initialize weights with random
        # values from a normal distribution (np.random.randn).
        self.W = np.random.randn(in_features, out_features)

        # Define the layer's bias vector b.

        # If bias=True, initialize the vector b with random
        # values from a normal distribution (np.random.randn).
        if bias:
            self.b = np.random.randn(out_features)
        # If bias=False, initialize the vector b with zeros.
        else:
            self.b = np.zeros(out_features)

    def forward(self, x):
        '''
        Method implementing the forward pass of the layer.

        params:
            x (np.array): input batch to the layer.
                          Shape: (n_elems, in_features), where
                          - n_elems: number of elements in the batch,
                          - in_features: dimensionality of each element in the batch.
        returns:
            (np.array): layer output. Shape: (n_elems, out_features), where
                        - n_elems: number of elements in the input batch,
                        - out_features: dimensionality of each element after passing through the layer.
        '''

        # Compute the output of the layer
        # output = XW+b
        self.x = x
        return x @ self.W + self.b

    def backward(self, grad_output, learning_rate):
        '''
        Method implementing the backward pass of the layer,
        i.e., calculating gradients for the layer's parameters and updating them using gradient descent.

        params:
            grad_output (np.array): gradient from the next layer
                                    (or from the loss function if this is the last layer),
                                    i.e., dL/dy, where y is the output of this layer.
                                    Shape: (n_elems, out_features), where
                                    - n_elems: number of elements in the batch,
                                    - out_features: dimensionality of each output element.
            learning_rate (float): learning rate for gradient descent.

        returns:
            (np.array): gradient to be passed to the previous layer.
                        i.e., dL/dx, where x is the input to this layer.
                        Shape: (n_elems, in_features), where
                        - n_elems: number of elements in the input batch,
                        - in_features: dimensionality of each input element.
        '''


        # Compute full gradients for W, b, and x
        grad_W = np.dot(self.x.T, grad_output)
        grad_b = # <YOUR CODE HERE>
        grad_x = # <YOUR CODE HERE>

        # Ensure dimensions are correct
        assert grad_W.shape == self.W.shape and grad_b.shape == self.b.shape

        # Gradient descent step: update layer parameters
        self.W = self.W - learning_rate * grad_W
        self.b = self.b - learning_rate * grad_b

        return grad_x

The following cell checks whether the `backward` method of the `Linear` class works correctly for various parameter values.

In [ ]:
np.random.seed(42)

fc = Linear(3, 2)

# Input of 3 elements, each of size 3
X = np.array([[1, 3, 4], [1, -5, 6], [-3, 5, 3]])
# Get the network output
output = fc(X)

# Set the gradient from the previous layer
grad_output = np.array([[0.55, 0.45], [2.34, 8.3], [-1.21, -0.9]])
# Compute the layer gradient
grads = fc.backward(grad_output, learning_rate=0.001)

# Correct gradients
true_grads = np.array(([[0.21097385, 1.04159213, -0.23414599],
                        [0.01471742, 14.15673899, -2.49125564],
                        [-0.47658625, -2.15443, 0.49404884]]))

# Verify the backward method produces correct gradients
assert np.allclose(true_grads, grads), "Something is wrong; the backward method returns incorrect values"

### Task 2: Backward Pass for Sigmoid Activation Function

Let's define the `Sigmoid` class. You need to fill in the missing parts of the `_sigmoid` and `forward` methods with the code you implemented earlier, and implement the `_sigmoid_grad` and `backward` methods.

Again, **note** that in the `forward` method, we added a line to save the input vector into the variable `self.x`. This will be useful for computing the gradients of the sigmoid function in the `backward` method.

**Note:** In this class, the `backward` function only computes gradients, no gradient step is performed. That's because there are no trainable parameters in the sigmoid layer.

**A math expercise for you:** Derive how to get $\frac{\partial L}{\partial X}$ from $\frac{\partial L}{\partial Z}$, where $Z = \sigma(X)$ (an elementwise nonlinearity).

In [ ]:
class Sigmoid(Module):
    '''
    Class implementing the sigmoid activation function.

    This class does not need to override the `__init__` method because
    activation functions do not have trainable parameters that need to be defined.
    '''

    def _sigmoid(self, x):
        """
        Helper method implementing the sigmoid formula.

        params:
            x (np.ndarray): input batch.
                          Shape: (n_elems, n_features), where
                          - n_elems: number of elements in the batch,
                          - n_features: dimensionality of each element in the batch.
        returns:
              (np.ndarray): batch elements after applying the sigmoid function to each element.
                          Shape: (n_elems, n_features), where
                          - n_elems: number of elements in the input batch,
                          - n_features: dimensionality of each element in the batch.
        """
        return 1. / (1 + np.exp(-x))


    def forward(self, x):
        """
        Method implementing the forward pass of the layer.

        params:
            x (np.ndarray): input batch.
                          Shape: (n_elems, in_features), where
                          - n_elems: number of elements in the batch,
                          - in_features: dimensionality of each element in the batch.
        returns:
              (np.ndarray): batch elements after applying the activation function to each element.
                          Shape: (n_elems, in_features), where
                          - n_elems: number of elements in the input batch,
                          - in_features: dimensionality of each element in the batch.
        """
        self.x = x
        return self._sigmoid(x)

    def _sigmoid_grad(self, x):
        """
        Helper method implementing the gradient formula of the sigmoid function.

        params:
            x (np.ndarray): input batch.
                          Shape: (n_elems, in_features), where
                          - n_elems: number of elements in the batch,
                          - in_features: dimensionality of each element in the batch.
        returns:
              (np.ndarray): batch gradients.
                          Shape: (n_elems, in_features), where
                          - n_elems: number of elements in the input batch,
                          - in_features: dimensionality of each element in the batch.
        """
        # Compute the gradient of the sigmoid function
        # Or check the Week 4 solution for that
        return # <YOUR CODE HERE>


    def backward(self, grad_output, learning_rate=None):
        '''
        Method implementing the backward pass of the sigmoid layer,
        i.e., calculating the gradient for the layer parameters and updating them using gradient descent.

        params:
            grad_output (np.array): gradient from the next layer
                                    (or from the loss function if this is the last layer),
                                    i.e., dL/dy, where y is the output of this layer.
                                    Shape: (n_elems, n_features), where
                                    - n_elems: number of elements in the batch,
                                    - n_features: dimensionality of each element in the batch.
            learning_rate (float): learning rate for gradient descent.

        returns:
            (np.array): gradient to be passed to the previous layer.
                        i.e., dL/dx, where x is the input to this layer.
                        Shape: (n_elems, n_features), where
                        - n_elems: number of elements in the input batch,
                        - n_features: dimensionality of each element in the batch.
        '''
        # Compute the gradient for the input `self.x` using the chain rule
        return # <YOUR CODE HERE>

The following cell checks whether the `backward` method of the `Sigmoid` class works correctly for various parameter values.

In [ ]:
np.random.seed(42)

# Instantiate the sigmoid class
sigma = Sigmoid()

# Input batch of size (5, 2)
dummy_input = np.random.randn(5, 2)
# Forward pass through the sigmoid
output = sigma(dummy_input)

# Backward pass through the sigmoid
grads = sigma.backward(np.random.randn(5, 2))

# Correct gradient values
true_grads = np.array([[-0.10899228, -0.11587775],
                       [ 0.05456517, -0.28119315],
                       [-0.42537221, -0.13866281],
                       [-0.14351594,  0.06804195],
                       [-0.21494318, -0.32831568]])

# Verify the gradient is computed correctly
assert np.allclose(true_grads, grads), "Something is wrong; the backward method returns incorrect values"

### Task 3: Backward Pass for a Fully Connected Neural Network

Below is the `NN` class for the neural network. You need to complete the code for the `__init__`, `forward`, and `backward` methods.

Note that in this task, we changed how the layers are declared in the `__init__` method. All layers are stored in the list `self.layers`. This makes it easier to iterate over the network layers in the `forward` and `backward` methods.

In [ ]:
class NN(Module):

    def __init__(self):
        '''
        Method that defines the architecture of the neural network—its layers.
        '''

        # Define the network layers
        self.layers = [
            Linear(300, 200),
            Sigmoid(),

            # Define two additional layers:
            # 1. A fully connected layer with size (200, 1)
            Linear(200, 1),
            # 2. A sigmoid activation function
            Sigmoid()
        ]

    def forward(self, x):
        '''
        Method implementing the forward pass of the network.

        params:
            x (np.ndarray): input batch to the network.
                          Shape: (n_elems, in_features), where
                          - n_elems: number of elements in the batch,
                          - in_features: dimensionality of each element in the batch.
        returns:
              (np.ndarray): network output. Shape: (n_elems, out_features), where
                          - n_elems: number of elements in the input batch,
                          - out_features: dimensionality of each element after passing through the network layers.
        '''

        # Pass the input `x` through the network layers
        for layer in self.layers:
            x = layer(x)

        return x

    def backward(self, grad_output, learning_rate=0.001):
        '''
        Method implementing the backward pass of the network,
        invoking the `.backward()` method for the layers, starting from the last one.

        params:
            grad_output (np.array): gradient from the loss function (or elsewhere),
                                    i.e., dL/dy, where y is the network's output.
                                    Shape: (n_elems, out_features), where
                                    - n_elems: number of elements in the batch,
                                    - out_features: dimensionality of each element in the batch.
            learning_rate (float): learning rate for gradient descent.

        returns:
            (np.array): gradient to be passed to the previous layer.
                        i.e., dL/dx, where x is the input to the network.
                        Shape: (n_elems, in_features), where
                        - n_elems: number of elements in the input batch,
                        - in_features: dimensionality of each element in the input batch.
        '''

        # Pass the gradient through the network layers
        # by calling the `.backward()` method for each layer in reverse order.
        # <YOUR CODE HERE>

        return grad_output

The following cell checks that the `backward` method of the `NN` class works correctly for various parameter values.

In [ ]:
np.random.seed(42)

# Instantiate the model
model = NN()

# Create a random input vector of size (10, 300)
dummy_input = np.random.randn(2, 300)

# Forward pass through the model
output = model(dummy_input)

# Backward pass through the model
grad = model.backward(np.random.randn(2, 1), 0.001)

assert grad.shape == (2, 300), "Something is wrong; the backward method returns a gradient with incorrect dimensions"

true_grads_begin = np.array([[ 5.62780811e-02, -3.75732270e-03,  2.10385197e-02],
                               [-1.10893855e-08, -9.20973003e-09,  1.65502526e-08]])

true_grads_end = np.array([[ 3.38621490e-02,  7.01416822e-02,  1.75411821e-02],
                           [-4.03032637e-09,  1.48369458e-08, -5.74383689e-09]])

assert np.allclose(true_grads_begin, grad[:, :3]), "Something is wrong; the backward method returns incorrect values"
assert np.allclose(true_grads_end, grad[:, -3:]), "Something is wrong; the backward method returns incorrect values"

## Loss Function

### Task 4: Implementation of LogLoss

In this task, you need to implement the log loss function. We will use this loss function later to train our neural network.

LogLoss formula:

$$LogLoss = - \sum_{i=1}^N \left( y_i \cdot log(\widehat{y}_i + \epsilon) + (1-y_i) \cdot log(1 - \widehat{y}_i + \epsilon) \right)$$

Here:
- $N$: number of elements in the batch;
- $y_i$: ground truth label for the i-th element of the dataset, taking values 0 or 1 (as we are solving a binary classification task);
- $\widehat{y}_i$: model output for the i-th element of the dataset, taking values in the range [0, 1];
- $\epsilon$: a small value (e.g., 1e-5) for numerical stability. Without $\epsilon$, $\widehat{y}_i$ or $1 - \widehat{y}_i$ could be zero (or very close to zero), causing $log(\widehat{y}_i)$ or $log(1-\widehat{y}_i)$ to become extremely large. Adding $\epsilon$ prevents this.

Gradient formula:

$$\frac{\partial LogLoss}{\partial \widehat{y}_i} = -\frac{y_i}{\widehat{y}_i + \epsilon} + \frac{1-y_i}{1 - \widehat{y}_i + \epsilon}$$

We will implement the loss function as a class rather than a function. This approach has several advantages:
- Loss functions often have hyperparameters set once before training. These parameters can be conveniently defined in the class initializer (`__init__`). If implemented as a function, we would need to pass these hyperparameters every time the loss is called.
- Loss functions may also have internal parameters updated during network training. While `LogLoss` has no such parameters, other loss functions might.
- Implementing the loss function as a class aligns with how network layers are implemented. By inheriting from `Module` and implementing `forward` and `backward` methods, the loss function becomes conceptually similar to network layers. This makes the workflow consistent and intuitive.

In [ ]:
class LogLoss(Module):
    def __init__(self):
        '''
        Method defining hyperparameters for the loss function.
        '''
        self.eps = 1e-5

    def forward(self, y_pred, y):
        '''
        Method implementing the forward pass (calculation) of the loss function.

        params:
            y_pred (np.ndarray): vector of size (N) or (N, 1), model predictions.
                                 N: number of elements passed to the model input.
                                 Contains float values in the range [0, 1].
            y (np.ndarray):      vector of size (N) or (N, 1), ground truth labels for
                                 the elements in `y_pred`.
                                 Contains integer values from the set {0, 1}.
        returns:
              float: loss value.
        '''
        # Store `y_pred` for gradient computation in `backward`
        self.y_pred = np.array(y_pred)
        if len(self.y_pred.shape) < 2:
            self.y_pred = self.y_pred.reshape(-1, 1)

        # Store `y` for gradient computation in `backward`
        self.y = np.array(y)
        if len(self.y.shape) < 2:
            self.y = self.y.reshape(-1, 1)

        # Compute log loss between `self.y_pred` and `self.y`
        # Add `self.eps` for numerical stability
        # <YOUR CODE HERE>

        return loss

    def __call__(self, y_pred, y):
        # Override `__call__` method as the `forward` method now takes two inputs: `y_pred` and `y`.
        return self.forward(y_pred, y)

    def backward(self):
        '''
        Method computing the gradient of the log loss w.r.t. the model predictions `self.y_pred`.

        returns:
            (np.array): gradient to be passed to the previous layer.
                        Shape: (n_elems, n_features), where
                        - n_elems: number of elements in the input batch,
                        - n_features: dimensionality of each output element.
                                      (n_features=1 for binary classification in our case)
        '''
        # <YOUR CODE HERE>
        return grad

The following cell checks that the `forward` and `backward` methods of the `LogLoss` class work correctly.

In [ ]:
# Instantiate the loss function class
logloss = LogLoss()

# Compute the loss value for given `y_pred` and `y` (forward pass)
loss = logloss([0.56, 0.01, 0.23], [1, 0, 1])
assert np.allclose(loss, 2.0494231310627455), "Something is wrong: the forward method is incorrect"

# Compute the gradient
grad = logloss.backward()
assert grad.shape == (3, 1), "Something is wrong: the backward method returns a gradient with incorrect dimensions"

true_grads = np.array([[-1.78571429], [ 1.01010101], [-4.34782609]])
assert np.allclose(true_grads, grad), "Something is wrong: the backward method returns incorrect values"

## Training the Network

In this task, we will write the training function for our custom neural network.

First, let's create the dataset on which the neural network will be trained. We will generate 1,200 random points distributed around two centers. These points will belong to two different classes. We will use the `make_blobs` function from `sklearn.datasets` (https://www.google.com/search?q=make+blobs+sklearn).

In [ ]:
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt

# Generate 1,200 points distributed around two centers: (-2, 0.5) and (3, -0.5)
X, y = make_blobs(n_samples=1200, centers=[[-2, 0.5], [3, -0.5]], cluster_std=1, random_state=42)

# Plot the points on a plane, coloring them based on their class
colors = ("red", "green")
colored_y = np.zeros(y.size, dtype=str)

for i, cl in enumerate([0, 1]):
    colored_y[y.ravel() == cl] = str(colors[i])

plt.figure(figsize=(15, 10))
plt.scatter(X[:, 0], X[:, 1], c=colored_y)
plt.show()

We have created a dataset— a pair `(X, y)`, where `X` is an array of point coordinates, and `y` is the class label corresponding to each point in `X` (0 or 1).

Each element of `X` is a vector of size 2 (coordinates of a point on the plane `[x, y]`).

Next, we split our dataset `(X, y)` into training and test sets. The training set will contain 1,000 elements, and the test set will contain 200.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=200)

Now, we need to declare a neural network that accepts inputs of size 2.

In [ ]:
class NN(Module):

    def __init__(self):
        self.layers = [
            # Define 4 layers: two linear layers and two sigmoid activation functions.
            # The number of neurons in the hidden layer can be chosen freely,
            # but keep it under 10 for faster training.
            Linear(2, 2),
            Sigmoid(),
            Linear(2, 1),
            Sigmoid()
        ]

    def forward(self, x):
        # Implement the forward method. You can copy it from the NN class
        # we implemented earlier in this notebook.
        for layer in self.layers:
            x = layer(x)

        return x

    def backward(self, grad_output, learning_rate=0.001):
        # Implement the backward method. You can copy it from the NN class
        # we implemented earlier in this notebook.
        for layer in reversed(self.layers):
            grad_output = layer.backward(grad_output, learning_rate)

Initialize the neural network and the loss function:

In [ ]:
model = NN()
logloss = LogLoss()

Next, we will write the training code for our `model`. For now, we will implement standard gradient descent (not stochastic). We will implement stochastic gradient descent later.

In [ ]:
def train(model, loss_fn, X, y, epochs=10, lr=0.1):
    '''
    Function to train the network.
    params:
        model (class):   the neural network to be trained
        loss_fn (class): the loss function used for training
        X (np.ndarray):  array of feature data for training.
                         Shape: (n_elems, n_features), where
                              - n_elems: number of data elements
                              - n_features: dimensionality of each data element
        y (np.ndarray):  array of target variable values for elements in `X`.
                         Shape: (n_elems, n_out), where
                              - n_elems: number of data elements
                              - n_out: dimensionality of each target variable
                                (n_out=1 in our case since we are solving a binary classification task)
        epochs (int):    number of training epochs
        lr (float):      learning rate for training the network
    returns:
        model (class):     the trained neural network
        losses (np.array): array of loss values after each training epoch
    '''

    # Create an array to store loss values for the training set after each epoch
    losses = []

    # Loop through the training epochs
    for i in range(epochs):

        # Get the model's predictions for `X`
        y_pred = model(X)

        # Compute the loss between `y_pred` and the ground truth `y`
        loss = loss_fn(y_pred, y)

        # Store the loss value
        losses.append(loss)

        # Compute the gradient of the loss function
        loss_grad = loss_fn.backward()

        # Perform a training step for the network
        # (Don't forget to pass `lr` to the network's `.backward` function)
        model.backward(loss_grad, learning_rate=lr)

    return losses

Call the training function for our `model` and `logloss`:

In [ ]:
losses_train = train(model, logloss, X_train, y_train, epochs=20, lr=0.001)

Visualize how the loss changed during the training epochs:

In [ ]:
plt.plot(list(range(len(losses_train))), losses_train)
plt.title('Loss Change on Training Set During Training')
plt.xlabel('Epoch')
plt.ylabel('LogLoss')

Let’s visualize how well the network learned to separate the classes of points. We will plot the points of the test dataset and color regions of the plane based on the class assigned to the points in that region by the neural network.

In [ ]:
from matplotlib.colors import ListedColormap

plt.figure(figsize=(15, 8))

eps = 0.1
xx, yy = np.meshgrid(np.linspace(np.min(X_test[:, 0]) - eps, np.max(X_test[:, 0]) + eps, 200),
                     np.linspace(np.min(X_test[:, 1]) - eps, np.max(X_test[:, 1]) + eps, 200))
Z = model(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA'])
plt.pcolormesh(xx, yy, Z, cmap=cmap_light)

colored_y = np.zeros(y_test.size, dtype=str)
for i, cl in enumerate([0, 1]):
    colored_y[y_test.ravel() == cl] = str(colors[i])

plt.scatter(X_test[:, 0], X_test[:, 1], c=colored_y)

Compute the log loss and accuracy on the training and test sets by comparing the network's predictions to the ground truth labels.

Note that the network outputs the probability of an element belonging to the first class. To compute accuracy, we need to threshold this probability at 0.5.

In [ ]:
from sklearn.metrics import accuracy_score, log_loss

y_pred_train = model(X_train)
y_pred_test = model(X_test)

print('LogLoss on Training Set:', log_loss(y_train, y_pred_train))
print('LogLoss on Test Set:', log_loss(y_test, y_pred_test), end='\n\n')

print('Accuracy on Training Set:', accuracy_score(y_train, (y_pred_train > 0.5).astype(int)))
print('Accuracy on Test Set:', accuracy_score(y_test, (y_pred_test > 0.5).astype(int)))

If your code works correctly, the accuracy on both sets should be >0.95.

And that's it! We've successfully implemented our own neural network and trained it =)

Now you can do even more: implement your own dataloader that will generate batches and train your network with SGD rather than vanilla gradient descent. And you can also train your network on MNIST data.

## Additional: Stochastic Gradient Descent

In this task, we will implement mini-batch gradient descent.

We will implement a `DataLoader` class that contains the data elements and a `generate_batches` method. This method will shuffle the dataset and return mini-batches of a specified size.

The `generate_batches` method will be a generator. If you are not familiar with generators in Python, you can read more here:
- https://habr.com/ru/articles/132554/
- https://medium.com/flux-it-thoughts/python-generators-and-yield-8ecbdab34c40

In [ ]:
class DataLoader():
    '''
    Class implementing the generation of data batches in a loop
    '''

    def __init__(self, X, y, batch_size):
        '''
        params:
            X (np.ndarray):  array of feature data for training elements.
            y (np.ndarray):  array of target variable values for elements in `X`.
            batch_size (int): size of the mini-batch for gradient descent
        '''

        # Store the data
        self.X = np.array(X)
        self.y = np.array(y)

        # Ensure data consistency
        assert len(X) == len(y)

        # Store batch size and total number of data elements
        self.num_elements = len(X)
        self.batch_size = batch_size

    def generate_batches(self):
        '''
        Generator method for batches of size `self.batch_size`.
        The data is shuffled before batch generation.
        '''

        # Set seed for reproducibility of batch generation
        np.random.seed(42)

        # Shuffle the data before generating batches
        perm = np.random.permutation(self.num_elements)
        self.X = self.X[perm]
        self.y = self.y[perm]

        # Batch generation loop. Each iteration of the loop yields the next batch
        # of feature data and target variables of size `self.batch_size`.
        # If the number of data elements is not divisible by `self.batch_size`,
        # the last batch will be smaller.
        for num_batch in range(self.num_elements // self.batch_size + 1):
            curr_batch_X = self.X[num_batch * self.batch_size: (num_batch + 1) * self.batch_size]
            curr_batch_y = self.y[num_batch * self.batch_size: (num_batch + 1) * self.batch_size]
            yield curr_batch_X, curr_batch_y

    def __call__(self):
        '''
        This method is called when an instance of the `DataLoader` class is invoked.
        For example:
            dataloader = DataLoader()
            dataloader()
        When `dataloader()` is called, the `__call__` method is invoked.

        Here, we define that calling the `__call__` method invokes `generate_batches`.
        i.e., `dataloader()` is equivalent to `dataloader.generate_batches()`.
        '''
        return self.generate_batches()

    def __len__(self):
        '''
        This method defines the `len` function for `DataLoader`.
        For example:
            dataloader = DataLoader()
            x = len(dataloader)
        The value of `x` will be the result of the `__len__` method.
        '''
        return self.num_elements

Batches can be generated from this `DataLoader` as follows:

'''
for X_batch, y_batch in dataloader():
    ...
'''

Each iteration of the loop assigns the outputs of `yield` (i.e., `curr_batch_X`, `curr_batch_y`) to `X_batch`, `y_batch`.

The code below verifies that the `DataLoader` class is implemented correctly:

In [ ]:
# Generate 100 data elements
X_fake = np.arange(100)[:, np.newaxis]
y_fake = np.arange(100) + 1000

# Initialize the DataLoader class
dataloader = DataLoader(X_fake, y_fake, batch_size=7)

# Generate data batches from `dataloader` and store in arrays
X_reconstructed, y_reconstructed = [], []
batch_num = 0
for X_batch, y_batch in dataloader():
    X_reconstructed.append(X_batch)
    y_reconstructed.append(y_batch)
    batch_num += 1

X_reconstructed = np.concatenate(X_reconstructed)
y_reconstructed = np.concatenate(y_reconstructed)

# Verify the number of batches
assert batch_num == 100 // 7 + 1, "Something is wrong; the number of batches is incorrect"

# Verify that during one epoch (one call to `generate_batches`),
# all data elements appear in a batch exactly once
assert (np.sort(X_reconstructed, axis=0) == X_fake).all(), \
            "Something is wrong; the sets of elements in `X_fake` and `X_reconstructed` do not match"

assert (np.sort(y_reconstructed, axis=0) == y_fake).all(), \
            "Something is wrong; the sets of elements in `y_fake` and `y_reconstructed` do not match"

# Verify that the data is shuffled before batch generation
assert (X_fake != X_reconstructed).all(), \
            "Something is wrong; the data is not shuffled"
assert (y_fake != y_reconstructed).all(), \
            "Something is wrong; the data is not shuffled"

Now, let's update our training function so that it works with our batch generator (`dataloader`):

In [ ]:
def train_stochastic(model, loss_fn, dataloader, epochs=10, lr=0.1):
    '''
    Network training function.
    params:
        model (class):   the neural network to be trained
        loss_fn (class): the loss function used for training
        dataloader (class DataLoader): batch generator for training the network
        epochs (int):    number of training epochs
        lr (float):      learning rate for training the network
    returns:
        model (class):     trained neural network
        losses (np.array): array of loss function values after each training epoch
    '''

    # Create an array to store the loss values on the training set
    # after each training epoch
    losses = []

    # Training epochs loop
    for i in range(epochs):
        for X_batch, y_batch in dataloader():

            # Get the model's predictions for `X`
            y_pred = model(X_batch)

            # Compute the loss between `y_pred` and the true labels `y`
            loss = loss_fn(y_pred, y_batch)

            # Store the loss value
            losses.append(loss)

            # Compute the gradient of the loss function
            loss_grad = loss_fn.backward()

            # Perform a training step
            # (Don’t forget to pass the learning rate `lr` to the `.backward` method)
            model.backward(loss_grad, learning_rate=lr)

    return losses

Finally, let's train our neural network using stochastic gradient descent:

In [ ]:
# Initialize the model, loss function, and dataloader
model = NN()
logloss = LogLoss()
dataloader = DataLoader(X, y, batch_size=20)

In [ ]:
# Train the neural network
losses_train = train_stochastic(model, logloss, dataloader, epochs=20, lr=0.001)

Let's again visualize how the loss function changed during training:

In [ ]:
plt.plot(list(range(len(losses_train))), losses_train)
plt.title('Loss Change on Training Set During Training')
plt.xlabel('Epoch')
plt.ylabel('LogLoss')

If the code is implemented correctly, the graph above will show a more "jagged" curve compared to the previous one when we trained the network with regular gradient descent. This is because, at each iteration, the loss function is computed over a mini-batch, and the characteristics of different batches can vary significantly. However, the loss generally decreases during training.

Next, let's visualize how well our network has learned to separate the point classes. As before, we will visualize the points from the test dataset on a plane and color the regions based on the class assigned to the points in that region by our neural network.

In [ ]:
from matplotlib.colors import ListedColormap

plt.figure(figsize=(15, 8))

eps = 0.1
xx, yy = np.meshgrid(np.linspace(np.min(X_test[:, 0]) - eps, np.max(X_test[:, 0]) + eps, 200),
                     np.linspace(np.min(X_test[:, 1]) - eps, np.max(X_test[:, 1]) + eps, 200))
Z = model(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA'])
plt.pcolormesh(xx, yy, Z, cmap=cmap_light)

colored_y = np.zeros(y_test.size, dtype=str)
for i, cl in enumerate([0, 1]):
    colored_y[y_test.ravel() == cl] = str(colors[i])

plt.scatter(X_test[:, 0], X_test[:, 1], c=colored_y)

Congratulations, the tasks are complete!

At this stage, we have implemented all the main components of a framework for working with neural networks. Next, you will find instructions on how to transfer code from a Jupyter notebook into `.py` files and split them into modules to create a complete framework from the components. Splitting the code into modules is not a course requirement and is not graded, but you can do this to present the code neatly on your GitHub/GitLab.

Below, in this notebook, you can test the framework components you just wrote on a more complex dataset—MNIST.

## Additional: Testing on MNIST

In this section, we will train and test our neural network on the MNIST dataset.

MNIST is a dataset of black-and-white images of handwritten digits, each of size 28×28. The images are divided into 10 classes based on the digit shown in the image. The images look something like this:

We will use images of two classes—0 and 1—and train the network to classify them.

First, the MNIST dataset needs to be downloaded. The cell below does that:

In [ ]:
! pip install wldhx.yadisk-direct
! curl -L $(yadisk-direct https://disk.yandex.com/d/rWEouTV8E1F-CA) -o MNIST_csv.zip
! unzip -qq MNIST_csv.zip

The dataset we downloaded contains images not as `.jpg` or `.png` files but as a `.csv` table. Each row in the table represents one 28×28 image flattened into a vector.

The data is already split into training and test sets. The training set is in `mnist_train.csv`, and the test set is in `mnist_test.csv`.

Let's load the training set and examine the data:

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('./MNIST_csv/mnist_train.csv')
data.head()

The column `label` represents the class number of the image (e.g., `label=7` means the corresponding image depicts the digit 7). The next 784 columns contain values from 0 to 255, describing all the pixels of the image.

Let's visualize the image from row 0:

In [ ]:
import matplotlib.pyplot as plt

img = np.array(data.iloc[0][1:])
plt.imshow(img.reshape(28, 28))

Split the data into images (`X`) and the target variable (`y`). Also, divide all pixel values by 255 so they fall in the range [0, 1]. Why normalizing data is important for training neural networks will be discussed in the next module.

In [ ]:
X = data.drop('label', axis=1) / 255
y = data['label']

Select only the images depicting 0 and 1. We will solve a binary classification task using these images.

In [ ]:
X = X[(y == 0) | (y == 1)]
y = y[(y == 0) | (y == 1)]

Implement a neural network class for working with images:

In [ ]:
class NN(Module):

    def __init__(self):
        self.layers = [
            # Define the network layers. Think about how many neurons
            # should be in the first and last layers. The total number
            # of layers and the number of neurons in hidden layers can be chosen freely.
            # However, keep the network small for faster training.
            Linear(784, 256),
            Sigmoid(),
            Linear(256, 1),
            Sigmoid()
        ]

    def forward(self, x):
        # Implement the forward method. You can copy it from the NN class
        # we implemented earlier in this notebook.
        for layer in self.layers:
            x = layer(x)

        return x

    def backward(self, grad_output, learning_rate=0.001):
        # Implement the backward method. You can copy it from the NN class
        # we implemented earlier in this notebook.
        # <YOUR CODE HERE>

Initialize the network, loss function, and dataloader:

In [ ]:
model = NN()
logloss = LogLoss()
mnist_dataloader = DataLoader(X, y, batch_size=100)

Train the neural network on the training set using stochastic gradient descent:

In [ ]:
loss_history = train_stochastic(model, logloss, mnist_dataloader, epochs=5, lr=0.0001)

Visualize the loss function change during training:

In [ ]:
plt.plot(list(range(len(loss_history))), loss_history)

Great, the network is trained. Now, load the MNIST test data and test the network on it:

In [ ]:
test_data = pd.read_csv('./MNIST_csv/mnist_test.csv')
test_data.head()

Preprocess the data:
- Split into images and target variable;
- Divide pixel values by 255;
- Keep only images of classes 0 and 1.

In [ ]:
X_test = test_data.drop('label', axis=1) / 255
y_test = test_data['label']

# Keep only images depicting 0 or 1
X_test = X_test[(y_test == 0) | (y_test == 1)]
y_test = y_test[(y_test == 0) | (y_test == 1)]

Initialize the test dataloader:

In [ ]:
test_loader = DataLoader(X_test, y_test, batch_size=100)

Iterate over the batches from the test dataloader, get the network's output for each batch, and save it in the `y_preds` array. Also, save the true labels for the batch in the `y_tests` array:

In [ ]:
y_preds = []
y_tests = []

for X_batch, y_batch in test_loader():
    y_pred = model(X_batch)
    y_preds.extend(y_pred.flatten())
    y_tests.extend(y_batch.flatten())

Calculate accuracy by comparing the true labels with the model's predictions. To do this, convert the model's predictions from probabilities of belonging to the first class into 0 or 1 values using a threshold of 0.5:

In [ ]:
y_preds = [1 if x > 0.5 else 0 for x in y_preds]

Finally, compute accuracy as the ratio of correct predictions to the total number of elements:

In [ ]:
np.sum(np.array(y_preds) == np.array(y_tests)) / len(y_preds)

Voilà! You have trained and tested your custom neural network on the MNIST dataset! =)